# Junction current–voltage characteristics

A direct tridiagonal minority-diffusion solve is validated before using the chapter API.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "figures" / "src").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate lecture-notes/figures/src")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "figures" / "src"))
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

length,D,tau,N=120e-6,1e-3,2e-6,201
x=np.linspace(0,length,N); dx=x[1]-x[0]; L=np.sqrt(D*tau)
u0=1.0
main=np.full(N-2,-2-dx**2/L**2)
A=diags((np.ones(N-3),main,np.ones(N-3)),(-1,0,1),format="csr")
b=np.zeros(N-2); b[0]=-u0
u=np.r_[u0,spsolve(A,b),0.0]
exact=np.sinh((length-x)/L)/np.sinh(length/L)
print(f"profile relative L2 error = {np.linalg.norm(u-exact)/np.linalg.norm(exact):.3e}")

profile relative L2 error = 4.027e-06


In [2]:
from semphy.junctions import minority_diffusion_fd

solution=minority_diffusion_fd(length,D,tau,u0,points=N)
flux_exact=D*u0/L/np.tanh(length/L)
flux_error=abs(solution["particle_flux_m2_s"]-flux_exact)/flux_exact
print(f"junction-flux relative error = {flux_error:.3e}")
assert np.linalg.norm(solution["excess_m3"]-exact)/np.linalg.norm(exact) < 1e-4
assert flux_error < 0.02

junction-flux relative error = 6.623e-03


In [3]:
# The ideal exponential has the expected voltage-per-decade slope.
from scipy.constants import elementary_charge as q, Boltzmann as k_B
from semphy.junctions import ideal_diode_current_density
T=300.0; v=np.array([0.4,0.4+(k_B*T/q)*np.log(10)])
j=ideal_diode_current_density(v,1e-8,T)
print(f"current ratio over one ideal decade = {j[1]/j[0]:.6f}")

current ratio over one ideal decade = 10.000002


In [4]:
from semphy.junctions import diode_current_figure
from semphy.junction_atlas import injection_transport_atlas, diode_characteristics_atlas, temperature_characteristics_atlas

diode_current_figure()
injection_transport_atlas()
diode_characteristics_atlas()
temperature_characteristics_atlas()
print("wrote current validation and three original diode-characteristic plates")

wrote current validation and three original diode-characteristic plates
